# Assignment: Noise to Picture

**Unit:** Deep Dive: Vision-Language Models | **Companion reading:** Generating Images by Removing Noise: Diffusion
**Suggested time:** about 1 hour

---

## What is this assignment trying to do?

In the lesson you saw that a diffusion model is trained by **adding noise** to pictures and learning to predict that noise, and that it makes new pictures by starting from pure noise and **removing** it a little at a time. You will write the three key pieces yourself:

1. **Part 1:** Add noise to a picture. *(TODO 1)*
2. **Part 2:** Write the training loss, then train a tiny diffusion model. It takes about a minute on a free GPU. *(TODO 2)*
3. **Part 3:** Write one noise-removal step, then generate new pictures. *(TODO 3)*

There are only **three short coding tasks** and **three short written reflections**. The reflections are a big part of your grade, because the goal is to understand what you see, not just to make the code run.

### How this assignment is graded

Your functions are **autograded**: the autograder runs them on its **own** small test models, so it never needs your trained model or a GPU. So:

- Keep every function **name and signature** exactly as given.
- Code outside the three graded functions (training, plots) is not run by the autograder.
- Your written reflections are also checked, and may be read by staff.

Fill in every `# TODO`, then run the cells below it to see your code at work.

---
## Setup

**Use a GPU.** In Colab: **Runtime → Change runtime type → T4 GPU**. Then run the next cell.

In [ ]:
# Colab already has PyTorch, NumPy, Pillow, and Matplotlib, so there is nothing to install.
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw

SEED = 0

import matplotlib.pyplot as plt

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("No GPU found. Training will be slow on a CPU, so switch to a T4 GPU if you can.")

---
## Part 1: Adding Noise to a Picture

A diffusion model learns from pictures that have been mixed with random noise. In the lesson, the **shortcut formula** gives the noisy picture at any step $t$ directly:

$$x_t = \sqrt{\bar\alpha_t}\; x_0 + \sqrt{1 - \bar\alpha_t}\; \epsilon$$

- $x_0$ is the clean picture and $\epsilon$ is a picture of random noise of the same shape.
- $\bar\alpha_t$ ("alpha bar") is a number between 0 and 1 for each step $t$. Near 1, almost all of the picture is kept. Near 0, almost nothing is kept. We give you a function `make_alpha_bar()` that builds these numbers for 1000 steps.

First, run the next two cells. They draw a small toy picture (numbers between -1 and 1, the range diffusion models use) and the 1000-step schedule.

In [ ]:
# Toy picture helpers (PROVIDED)
def make_scene(size=96):
    """Draw a small picture (sky, sun, hills, a house) as a PIL image."""
    img = Image.new("RGB", (size, size))
    d = ImageDraw.Draw(img)
    for y in range(size):                                     # sky
        f = y / size
        d.line([(0, y), (size, y)], fill=(int(120 + 90 * f), int(180 + 50 * f), 255))
    s = size / 96
    d.ellipse([60 * s, 10 * s, 82 * s, 32 * s], fill=(255, 220, 40))        # sun
    d.polygon([(0, 70 * s), (30 * s, 48 * s), (60 * s, 70 * s), (size, 60 * s), (size, size), (0, size)],
              fill=(60, 160, 80))                                              # hills
    d.rectangle([22 * s, 56 * s, 52 * s, 82 * s], fill=(200, 80, 60))         # house
    d.polygon([(18 * s, 56 * s), (37 * s, 40 * s), (56 * s, 56 * s)], fill=(120, 50, 40))
    d.rectangle([32 * s, 66 * s, 42 * s, 82 * s], fill=(250, 240, 200))       # door
    return img


def to_array(img):
    """PIL image -> float array with values in [-1, 1], shape (height, width, 3)."""
    return np.asarray(img, dtype=np.float32) / 127.5 - 1.0


def to_pil(x):
    """Float array with values near [-1, 1] -> PIL image (values outside the range are clipped)."""
    return Image.fromarray(np.clip((np.asarray(x) + 1.0) * 127.5, 0, 255).astype(np.uint8))

In [ ]:
# Provided helpers (PROVIDED)
def make_alpha_bar(T=1000, beta_start=1e-4, beta_end=0.02):
    """The noise schedule: a NumPy array of T numbers. alpha_bar[t] is how much picture is kept at step t."""
    betas = np.linspace(beta_start, beta_end, T)      # how much noise to add at each step
    return np.cumprod(1.0 - betas)                    # running product of (1 - beta)


def predict_clean(x_t, eps_hat, abar_t):
    """Undo the shortcut formula: from a noisy picture and a noise guess, guess the clean picture."""
    return (x_t - (1.0 - abar_t) ** 0.5 * eps_hat) / abar_t ** 0.5


def ddim_sample(model, x_T, alpha_bar, steps):
    """Turn the noise x_T into pictures by removing noise in `steps` steps (uses YOUR ddim_step)."""
    timesteps = np.linspace(len(alpha_bar) - 1, 0, steps).round().astype(int)    # noisiest -> cleanest
    x = x_T
    for i, t in enumerate(timesteps):
        abar_t = alpha_bar[int(t)]
        abar_prev = alpha_bar[int(timesteps[i + 1])] if i + 1 < len(timesteps) else 1.0   # 1.0 = no noise left
        t_batch = torch.full((x.shape[0],), int(t), dtype=torch.long, device=x.device)
        eps_hat = model(x, t_batch)                       # the network's guess of the noise
        x = ddim_step(eps_hat, x, abar_t, abar_prev)      # your one step
    return x

In [ ]:
scene = make_scene(96)
x0 = to_array(scene)                     # the clean picture, shape (96, 96, 3), values in [-1, 1]
T = 1000
alpha_bar = make_alpha_bar(T)            # alpha_bar[t] = how much picture is kept at step t
display(scene.resize((192, 192)))

### TODO 1: `add_noise(x0, abar_t, eps)`

Write the shortcut formula from above: return the noisy picture $x_t$.

It will be used on plain numbers, NumPy arrays, *and* PyTorch tensors.

In [ ]:
def add_noise(x0, abar_t, eps):
    """Mix the clean picture x0 with noise eps, as if t steps of noise had been added."""
    return None  # TODO 1

### Watch the picture disappear (PROVIDED)

The next cell shows the picture at six different steps. The title of each picture shows how much of it is kept ($\bar\alpha_t$).

In [ ]:
rng = np.random.default_rng(SEED)
eps = rng.standard_normal(x0.shape)

shown = [0, 50, 100, 250, 500, 999]
fig, axes = plt.subplots(1, len(shown), figsize=(16, 3.2))
for ax, t in zip(axes, shown):
    ax.imshow(to_pil(add_noise(x0, alpha_bar[t], eps)))
    ax.set_title(f"t = {t}\nalpha_bar = {alpha_bar[t]:.3f}")
    ax.axis("off")
plt.show()

### Part 1 reflection

Look at the six noisy pictures. Around which step does the scene stop being recognizable? In your own words, explain what a **small** $\bar\alpha_t$ means for the picture. Write at least 30 words, and use the words *noise* and *signal* (the part of the picture that is left).

In [ ]:
part1_reflection = """
TODO: your answer here
"""

---
## Part 2: Train a Tiny Diffusion Model

The network in a diffusion model does not draw a clean picture directly. It looks at a noisy picture and **predicts the noise** in it. Run the next cell to load 20,000 small 28 x 28 pictures of clothing (Fashion-MNIST), scaled to values between -1 and 1, and a small network called `TinyUNet`. You do not need to read or change the network: it takes a noisy picture `x_t` and the step number `t`, and returns its guess of the noise, `model(x_t, t)`.

In [ ]:
# The training pictures and the network (PROVIDED)
def show_images(x, ncols=8, title=""):
    """Show a batch of (N, 1, 28, 28) pictures with values in [-1, 1] as a grid."""
    x = x.detach().cpu().clamp(-1, 1)
    nrows = (len(x) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(1.2 * ncols, 1.2 * nrows + 0.4), squeeze=False)
    for k, ax in enumerate(axes.flat):
        ax.axis("off")
        if k < len(x):
            ax.imshow(x[k, 0], cmap="gray", vmin=-1, vmax=1)
    fig.suptitle(title)
    plt.show()


from torchvision import datasets
DATA = datasets.FashionMNIST("data", train=True, download=True).data[:20000]
DATA = DATA.float().div(127.5).sub(1).unsqueeze(1).to(DEVICE)       # shape (20000, 1, 28, 28), values in [-1, 1]
show_images(DATA[:16], title="Some training pictures")

In [ ]:
# The small network that will learn to predict noise (PROVIDED)
def timestep_embedding(t, dim):
    """Turn step numbers (a LongTensor of shape (B,)) into vectors of length `dim`."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
    angles = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)


class TinyUNet(nn.Module):
    """A small U-Net: model(x_t, t) -> predicted noise, same shape as x_t. Input: (B, 1, 28, 28)."""

    def __init__(self, ch=64, t_dim=128):
        super().__init__()
        self.t_dim = t_dim
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        self.inc = nn.Conv2d(1, ch, 3, padding=1)
        self.down1 = nn.Conv2d(ch, ch * 2, 4, stride=2, padding=1)               # 28 -> 14
        self.down2 = nn.Conv2d(ch * 2, ch * 4, 4, stride=2, padding=1)           # 14 -> 7
        self.mid = nn.Conv2d(ch * 4, ch * 4, 3, padding=1)
        self.up1 = nn.ConvTranspose2d(ch * 4, ch * 2, 4, stride=2, padding=1)    # 7 -> 14
        self.up2 = nn.ConvTranspose2d(ch * 4, ch, 4, stride=2, padding=1)        # 14 -> 28
        self.out = nn.Conv2d(ch * 2, 1, 3, padding=1)
        widths = [ch, ch * 2, ch * 4, ch * 4, ch * 2, ch]
        self.norms = nn.ModuleList([nn.GroupNorm(8, w) for w in widths])
        self.t_proj = nn.ModuleList([nn.Linear(t_dim, w) for w in widths])       # tells every layer the step
        self.act = nn.SiLU()

    def block(self, k, conv, h, emb):
        h = self.norms[k](conv(h)) + self.t_proj[k](emb)[:, :, None, None]
        return self.act(h)

    def forward(self, x, t):
        emb = self.t_mlp(timestep_embedding(t, self.t_dim))
        h0 = self.block(0, self.inc, x, emb)
        h1 = self.block(1, self.down1, h0, emb)
        h2 = self.block(2, self.down2, h1, emb)
        h2 = self.block(3, self.mid, h2, emb)
        u1 = self.block(4, self.up1, h2, emb)
        u2 = self.block(5, self.up2, torch.cat([u1, h1], dim=1), emb)
        return self.out(torch.cat([u2, h0], dim=1))

### TODO 2: `noise_prediction_loss(model, x0, t, eps, alpha_bar)`

For every training step we take a batch of clean pictures `x0`, pick a random step `t` for each and some random noise `eps`, and then measure how well the network can find that noise. Fill in the three blanks:

- **2a:** build the noisy pictures `x_t` with your `add_noise`. (`abar_t` is already computed for you: one number per picture.)
- **2b:** get the network's guess of the noise.
- **2c:** return the **mean squared error** between the true noise `eps` and the guess, as a single number.

In [ ]:
def noise_prediction_loss(model, x0, t, eps, alpha_bar):
    """Mean squared error between the true noise and the model's guess, for one batch."""
    abar_t = alpha_bar[t].view(-1, 1, 1, 1)   # provided
    x_t = None        # TODO 2a
    eps_hat = None    # TODO 2b
    return None       # TODO 2c

### Train it (PROVIDED)

This cell trains the network with your loss function. On a free T4 GPU it takes **about a minute** (on a CPU it only does one quick pass, so the pictures will be poor).

In [ ]:
ALPHA_BAR = torch.tensor(alpha_bar, dtype=torch.float32, device=DEVICE)

model = TinyUNet().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
EPOCHS = 5 if DEVICE == "cuda" else 1
BATCH = 128
losses = []

start = time.time()
for epoch in range(EPOCHS):
    order = torch.randperm(len(DATA), device=DEVICE)
    for i in range(0, len(DATA) - BATCH + 1, BATCH):
        x0_batch = DATA[order[i:i + BATCH]]
        t = torch.randint(0, T, (BATCH,), device=DEVICE)
        eps_batch = torch.randn_like(x0_batch)
        loss = noise_prediction_loss(model, x0_batch, t, eps_batch, ALPHA_BAR)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    print(f"epoch {epoch + 1}/{EPOCHS}: average loss {np.mean(losses[-100:]):.4f}   ({time.time() - start:.0f} s)")

model.eval()
plt.figure(figsize=(6, 3))
plt.plot(np.convolve(losses, np.ones(20) / 20, mode="valid"))
plt.xlabel("training step")
plt.ylabel("loss (smoothed)")
plt.show()

### Part 2 reflection

Describe the loss curve: where did it start, how did it change, and did it flatten out? The loss is the average squared error of the **noise** prediction. Why is this a sensible number to minimize, even though we want to generate *pictures*? Write at least 30 words, and use the word *noise*.

In [ ]:
part2_reflection = """
TODO: your answer here
"""

---
## Part 3: Generating New Pictures

To generate, start from a batch of pure noise and walk backward, removing a little noise at each step. This is the **DDIM** sampler. We give you the loop (`ddim_sample`, in the cell below). Your job is the **one step** that the loop repeats.

At each step the loop asks the network for its noise guess `eps_hat`, and then calls your function `ddim_step`. Fill in the two blanks:

- **3a:** keep the clean-picture guess `x0_hat` in the range [-1, 1]. (The guess itself is already written for you.)
- **3b:** move to the next, less noisy step: mix `x0_hat` with the noise guess again using your `add_noise`, with the **next** step's number `abar_prev`.

(`abar_prev` is `alpha_bar` at the next step. On the very last step it is `1.0`, which means "no noise left", so the result is just `x0_hat`.)

### TODO 3: `ddim_step(eps_hat, x, abar_t, abar_prev)`

In [ ]:
def ddim_step(eps_hat, x, abar_t, abar_prev):
    """One step: from the noisy picture x (with abar_t) to a less noisy one (with abar_prev)."""
    x0_hat = predict_clean(x, eps_hat, abar_t)   # provided
    x0_hat = None     # TODO 3a
    return None       # TODO 3b

### Generate and compare steps (PROVIDED)

We make one batch of starting noise and run the sampler with different numbers of steps **from the same noise**. The time is how long it took to turn that batch of noise into pictures.

In [ ]:
g = torch.Generator().manual_seed(SEED)
x_T = torch.randn(16, 1, 28, 28, generator=g).to(DEVICE)
with torch.no_grad():
    for n_steps in [2, 10, 50]:
        t0 = time.time()
        pics = ddim_sample(model, x_T, ALPHA_BAR, n_steps)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        show_images(pics, ncols=8, title=f"{n_steps} steps  ({time.time() - t0:.2f} s)")

### Part 3 reflection

Compare the three rows. What changed when you used more steps: the overall shape, the fine detail, or both? Did more steps always help, and what did they cost in time? Write at least 30 words, and use the words *step* and *noise*.

In [ ]:
part3_reflection = """
TODO: your answer here
"""

---
## Submitting

1. Make sure every `TODO` is complete, and all three reflections (`part1_reflection` to `part3_reflection`) are written.
2. Run all cells from top to bottom (**Runtime → Run all** in Colab).
3. Download the notebook as a Python file (**File → Download → Download .py**) and rename it `firstname_lastname_diffusion.py`.
4. Submit that `.py` file on Gradescope (see the assignment page on the course website for the link).

Your pictures are not submitted: the autograder checks your functions and your written answers.